# Июнь: месячная комиссия — отчёт эквайринга vs MPOS_RENT vs ЦФТ

Две задачи:
1. **ЦФТ только месячная комиссия** — не все виды `VID_COMISS`.
2. **Почему MPOS закрывает ~21% витрины** — гипотеза: **нулей в MPOS_RENT нет**, пишутся только ненулевые `n_amt`. Эталон 1:1 — отчёт `06_Июнь_2026.xlsx`.

| Источник | Что берём |
|---|---|
| Excel | `Комиссия (₽ в месяц)` |
| MPOS_RENT | `scd1_mrc_pos_rent.n_amt` за `d_rent` июня |
| ЦФТ | `DOG_OPER` + `VID_COMISS`, только виды, которые бьются с Excel-месячной |

Новый kernel нормален. Нужны Impala и файл Excel на `/home/jovyan/documents/Equaring/Data`.


In [ ]:
import re
from decimal import Decimal, InvalidOperation
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display
from rail_connectors.connection import connect

pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 140)
pd.set_option('display.width', 220)
pd.set_option('display.float_format', lambda x: f'{x:,.2f}'.replace(',', ' '))

DATA_DIR = Path('/home/jovyan/documents/Equaring/Data')
OUT_DIR = DATA_DIR / 'qc_june_mpos_excel_cft_monthly'
OUT_DIR.mkdir(parents=True, exist_ok=True)

MONTH = '2026-06'
MONTH_START = '2026-06-01'
MONTH_END = '2026-06-30'
MONTH_END_EXCL = '2026-07-01'
EXCEL_PATH = DATA_DIR / '06_Июнь_2026.xlsx'
EXCEL_HEADER = 0
VAT = 1.22
EPS = 0.01
MEM_LIMIT = '8g'

# Если автоотбор видов ЦФТ не устроит — пропиши точные имена из профиля
CFT_MONTHLY_TYPES_FORCE = []  # например: ['Ежемесячная комиссия']

print('MONTH:', MONTH)
print('EXCEL:', EXCEL_PATH, 'exists=', EXCEL_PATH.exists())
print('OUT_DIR:', OUT_DIR)


## 0) Helpers + Impala


In [ ]:
def normalize_inn_q1(v):
    if pd.isna(v):
        return None
    s = str(v).strip()
    s = re.sub(r'\.0$', '', s)
    s = re.sub(r'\D+', '', s)
    if not s:
        return None
    if len(s) == 9:
        s = s.zfill(10)
    elif len(s) == 11:
        s = s.zfill(12)
    return s if len(s) in (10, 12) else None


def normalize_agr_q1(v):
    if pd.isna(v):
        return None
    s = str(v).strip().replace('\xa0', '').replace(' ', '').replace(',', '.')
    if s in {'', 'nan', 'None'}:
        return None
    try:
        d = Decimal(s)
        if d == d.to_integral_value():
            return str(int(d))
    except (InvalidOperation, ValueError):
        pass
    s = re.sub(r'\.0$', '', s)
    return s if s not in {'', 'nan', 'None'} else None


def to_num(s):
    return pd.to_numeric(
        s.astype(str).str.replace('\xa0', '', regex=False).str.replace(' ', '', regex=False).str.replace(',', '.', regex=False),
        errors='coerce',
    )


def pick_col(columns, aliases):
    cols = list(columns)
    norm = lambda x: re.sub(r'\s+', ' ', str(x).replace('\xa0', ' ').replace('\n', ' ').strip().lower())
    nmap = {norm(c): c for c in cols}
    for a in aliases:
        if a in cols:
            return a
        if norm(a) in nmap:
            return nmap[norm(a)]
    for a in aliases:
        key = norm(a)
        for nk, orig in nmap.items():
            if key and key in nk:
                return orig
    return None


def fetch_imp(sql, label):
    t0 = pd.Timestamp.now()
    print(f'FETCH {label} ...')
    with imp:
        imp.execute(f'set MEM_LIMIT={MEM_LIMIT}')
        df = imp.fetch(sql)
    if df is None:
        df = pd.DataFrame()
    print(f'  rows={len(df):,}  {(pd.Timestamp.now() - t0).total_seconds():.1f}s')
    return df


def is_nz(s):
    return pd.to_numeric(s, errors='coerce').fillna(0).abs() >= EPS


if 'imp' in globals() and imp is not None:
    print('Reuse existing imp')
else:
    imp = connect(
        to='IMPALA',
        extra_options={'db': 'sandbox_ai'},
        driver_args={'tez.queue.name': 'ai'},
        kerberos={
            'keytab_path': '/home/jovyan/test_requests/tech.keytab',
            'use_credentials': True,
            'update_keytab': True,
        },
        user_params={'user_name': 'Shestopalov-VYur'},
    )
    imp._init_connection()
    print('Impala connected')


## 1) Отчёт эквайринга за июнь


In [ ]:
if not EXCEL_PATH.exists():
    raise FileNotFoundError(f'Нет файла отчёта: {EXCEL_PATH}')

raw_ex = pd.read_excel(EXCEL_PATH, header=EXCEL_HEADER)
print('Excel raw rows=', f'{len(raw_ex):,}', '| cols=', list(raw_ex.columns)[:25])

col_inn = pick_col(raw_ex.columns, ['ИНН', 'inn', 'c_inn'])
col_agr = pick_col(raw_ex.columns, ['ID договора', 'agr_id', 'abs_agr_id', 'ИД договора'])
col_mon = pick_col(raw_ex.columns, [
    'Комиссия (₽ в месяц)', 'Комиссия (руб в месяц)', 'Комиссия в месяц',
    'Комиссия CN (₽ в месяц)', 'commission_monthly',
])
col_tar = pick_col(raw_ex.columns, ['Тариф', 'Тарифный план', 'tariff_name'])
print('resolved:', {'inn': col_inn, 'agr': col_agr, 'monthly': col_mon, 'tariff': col_tar})
if None in (col_inn, col_agr, col_mon):
    raise RuntimeError(f'Не найдены колонки Excel. cols={list(raw_ex.columns)}')

ex = pd.DataFrame({
    'inn_key': raw_ex[col_inn].map(normalize_inn_q1),
    'agr_id_key': raw_ex[col_agr].map(normalize_agr_q1),
    'commission_excel': to_num(raw_ex[col_mon]),
    'tariff': raw_ex[col_tar] if col_tar else np.nan,
})
ex = (
    ex.dropna(subset=['agr_id_key'])
    .groupby(['inn_key', 'agr_id_key'], as_index=False)
    .agg(commission_excel=('commission_excel', 'max'), tariff=('tariff', 'first'))
)
ex['excel_nz'] = is_nz(ex['commission_excel'])
ex['excel_zero'] = ~ex['excel_nz']

n_ex = len(ex)
n_ex_nz = int(ex['excel_nz'].sum())
n_ex_z = int(ex['excel_zero'].sum())
print('=== Excel June monthly ===')
print(f'keys={n_ex:,}  nonzero={n_ex_nz:,} ({100.0 * n_ex_nz / n_ex:.2f}%)  zero={n_ex_z:,} ({100.0 * n_ex_z / n_ex:.2f}%)')
print('sum nonzero=', f"{ex.loc[ex['excel_nz'], 'commission_excel'].sum():,.2f}")
display(ex.groupby('excel_nz', as_index=False).agg(keys=('agr_id_key', 'nunique'), inn=('inn_key', 'nunique'), sum_comm=('commission_excel', 'sum')))


## 2) Сырой MPOS_RENT за июнь — пишут ли нули?

Если гипотеза верна: в таблице почти нет `n_amt = 0`, есть только ненулевые строки.


In [ ]:
sql_mpos_raw = f'''
select
  count(*) as rent_rows,
  count(distinct cast(c_nmrc as string)) as n_nmrc,
  sum(case when n_amt is null then 1 else 0 end) as n_amt_null,
  sum(case when n_amt is not null and cast(n_amt as double) = 0 then 1 else 0 end) as n_amt_zero,
  sum(case when n_amt is not null and cast(n_amt as double) <> 0 then 1 else 0 end) as n_amt_nonzero,
  sum(cast(n_amt as double)) as sum_n_amt
from ods_alpha.scd1_mrc_pos_rent
where c_nmrc is not null
  and coalesce(cast(ods_deleted_flg as string), '0') not in ('1', 'Y', 'y')
  and cast(d_rent as date) between cast('{MONTH_START}' as date) and cast('{MONTH_END}' as date)
'''
mpos_raw_stat = fetch_imp(sql_mpos_raw, 'MPOS raw June stats')
display(mpos_raw_stat)

row = mpos_raw_stat.iloc[0] if len(mpos_raw_stat) else None
if row is not None:
    z = float(row['n_amt_zero'] or 0)
    nz = float(row['n_amt_nonzero'] or 0)
    print('HYPOTHESIS zeros-not-written:', 'YES' if z == 0 and nz > 0 else 'NO / mixed')
    print(f'zero rows={z:,.0f}  nonzero rows={nz:,.0f}')


## 3) MPOS_RENT → inn + agr_id (как шаг 10m)


In [ ]:
sql_mpos = f'''
with rent_base as (
  select
    cast(c_nmrc as string) as c_nmrc,
    cast(d_rent as date) as d_rent_dt,
    cast(n_amt as double) as n_amt_num,
    cast(ods_commit_ts as timestamp) as ods_commit_ts,
    cast(ods_insert_ts as timestamp) as ods_insert_ts,
    cast(ods_op_csn as decimal(38, 0)) as ods_op_csn,
    coalesce(cast(ods_deleted_flg as string), '0') as ods_deleted_flg
  from ods_alpha.scd1_mrc_pos_rent
  where c_nmrc is not null
    and cast(d_rent as date) between cast('{MONTH_START}' as date) and cast('{MONTH_END}' as date)
),
rent_ranked as (
  select *,
    row_number() over (
      partition by c_nmrc, d_rent_dt
      order by coalesce(ods_commit_ts, ods_insert_ts) desc, ods_op_csn desc, ods_insert_ts desc
    ) as rn
  from rent_base
  where ods_deleted_flg not in ('1', 'Y', 'y')
),
rent_dedup as (
  select c_nmrc, d_rent_dt, n_amt_num from rent_ranked where rn = 1
),
terms_active as (
  select distinct
    cast(t.n_agr as string) as n_agr,
    cast(t.c_nmrc as string) as c_nmrc,
    cast(t.d_valid_from as date) as d_valid_from,
    cast(t.d_valid_to as date) as d_valid_to
  from ods_alpha.scd1_agr_terms t
  where coalesce(cast(t.ods_deleted_flg as string), '0') not in ('1', 'Y', 'y')
    and t.c_nmrc is not null
    and cast(t.d_valid_from as date) <= cast('{MONTH_END}' as date)
    and (t.d_valid_to is null or cast(t.d_valid_to as date) >= cast('{MONTH_START}' as date))
),
agreements_active as (
  select distinct
    cast(a.n_agr as string) as n_agr,
    cast(a.abs_agr_id as string) as agr_id,
    cast(a.n_cmp_client as string) as n_cmp_client,
    cast(a.d_valid_from as date) as d_valid_from,
    cast(a.d_valid_to as date) as d_valid_to
  from ods_alpha.scd1_agreements a
  where coalesce(cast(a.ods_deleted_flg as string), '0') not in ('1', 'Y', 'y')
    and upper(trim(cast(a.acq_class as string))) = 'SA'
    and cast(a.d_valid_from as date) <= cast('{MONTH_END}' as date)
    and (a.d_valid_to is null or cast(a.d_valid_to as date) >= cast('{MONTH_START}' as date))
),
companies_active as (
  select distinct
    cast(c.n_cmp as string) as n_cmp,
    regexp_replace(trim(cast(c.c_inn as string)), '[^0-9]', '') as inn_key
  from ods_alpha.scd1_companies c
  where coalesce(cast(c.ods_deleted_flg as string), '0') not in ('1', 'Y', 'y')
    and c.c_inn is not null
)
select
  c.inn_key as inn,
  cast(a.agr_id as string) as agr_id,
  count(*) as rent_rows,
  sum(r.n_amt_num) as commission_mpos,
  sum(case when r.n_amt_num is not null and r.n_amt_num = 0 then 1 else 0 end) as n_amt_zero,
  sum(case when r.n_amt_num is not null and r.n_amt_num <> 0 then 1 else 0 end) as n_amt_nonzero
from rent_dedup r
left join terms_active t
  on t.c_nmrc = r.c_nmrc
 and r.d_rent_dt between t.d_valid_from and coalesce(t.d_valid_to, cast('2999-12-31' as date))
left join agreements_active a
  on a.n_agr = t.n_agr
 and r.d_rent_dt between a.d_valid_from and coalesce(a.d_valid_to, cast('2999-12-31' as date))
left join companies_active c
  on c.n_cmp = a.n_cmp_client
group by 1, 2
'''

mpos_raw = fetch_imp(sql_mpos, 'MPOS mapped June')
mpos = mpos_raw.copy()
mpos['inn_key'] = mpos['inn'].map(normalize_inn_q1)
mpos['agr_id_key'] = mpos['agr_id'].map(normalize_agr_q1)
mpos['commission_mpos'] = pd.to_numeric(mpos['commission_mpos'], errors='coerce')
mpos_map = (
    mpos.dropna(subset=['agr_id_key'])
    .groupby(['inn_key', 'agr_id_key'], as_index=False)
    .agg(
        commission_mpos=('commission_mpos', 'sum'),
        rent_rows=('rent_rows', 'sum'),
        n_amt_zero=('n_amt_zero', 'sum'),
        n_amt_nonzero=('n_amt_nonzero', 'sum'),
    )
)
mpos_map['mpos_nz'] = is_nz(mpos_map['commission_mpos'])
print('MPOS mapped keys=', f'{len(mpos_map):,}', '| nonzero=', int(mpos_map['mpos_nz'].sum()))
print('MPOS unmapped groups=', int(mpos['agr_id_key'].isna().sum()))
print('sum mpos=', f"{mpos_map['commission_mpos'].fillna(0).sum():,.2f}")


## 4) Excel vs MPOS — проверка гипотезы нулей

Бакеты:
- Excel=0 и нет строки MPOS → нули просто не пишутся (гипотеза)
- Excel≠0 и есть MPOS → рабочий 1:1
- Excel≠0 и нет MPOS → настоящий недогруз
- Excel=0 и есть MPOS → MPOS пишет то, чего нет в отчёте


In [ ]:
cmp = ex.merge(
    mpos_map[['inn_key', 'agr_id_key', 'commission_mpos', 'mpos_nz', 'n_amt_zero', 'n_amt_nonzero']],
    on=['inn_key', 'agr_id_key'],
    how='outer',
    indicator='side',
)
cmp['commission_excel'] = pd.to_numeric(cmp['commission_excel'], errors='coerce')
cmp['commission_mpos'] = pd.to_numeric(cmp['commission_mpos'], errors='coerce')
cmp['excel_nz'] = is_nz(cmp['commission_excel'])
cmp['mpos_present'] = cmp['commission_mpos'].notna()
cmp['mpos_nz'] = is_nz(cmp['commission_mpos'])
cmp['delta'] = cmp['commission_mpos'].fillna(0) - cmp['commission_excel'].fillna(0)
cmp['exact'] = np.isclose(cmp['commission_mpos'].fillna(0), cmp['commission_excel'].fillna(0), atol=EPS, rtol=0)

def bucket(r):
    ez, mz, mp = r['excel_nz'], r['mpos_nz'], r['mpos_present']
    if (not ez) and (not mp):
        return 'excel_zero_no_mpos'
    if (not ez) and mp and (not mz):
        return 'excel_zero_mpos_zero'
    if (not ez) and mz:
        return 'excel_zero_mpos_nz'
    if ez and mz:
        return 'both_nz'
    if ez and mp and (not mz):
        return 'excel_nz_mpos_zero'
    if ez and (not mp):
        return 'excel_nz_no_mpos'
    return 'other'

cmp['bucket'] = cmp.apply(bucket, axis=1)

buck = (
    cmp.groupby('bucket', as_index=False)
    .agg(
        keys=('agr_id_key', 'nunique'),
        inn=('inn_key', 'nunique'),
        sum_excel=('commission_excel', 'sum'),
        sum_mpos=('commission_mpos', 'sum'),
    )
    .sort_values('keys', ascending=False)
)
print('=== Excel vs MPOS buckets ===')
display(buck)

n_ex_keys = ex['agr_id_key'].nunique()
n_mpos_in_ex = int(cmp.loc[cmp['agr_id_key'].isin(ex['agr_id_key']) & cmp['mpos_present'], 'agr_id_key'].nunique())
n_ex_nz_mpos = int(cmp.loc[cmp['excel_nz'] & cmp['mpos_present'], 'agr_id_key'].nunique())
n_ex_nz = int(ex.loc[ex['excel_nz'], 'agr_id_key'].nunique())
n_ex_z_nompos = int(cmp.loc[cmp['bucket'] == 'excel_zero_no_mpos', 'agr_id_key'].nunique())
n_ex_z = int(ex.loc[ex['excel_zero'], 'agr_id_key'].nunique())

print()
print(f'Excel keys: {n_ex_keys:,}')
print(f'MPOS present among Excel: {n_mpos_in_ex:,} ({100.0 * n_mpos_in_ex / n_ex_keys:.2f}%)')
print(f'Excel nonzero covered by MPOS: {n_ex_nz_mpos:,} / {n_ex_nz:,} = {100.0 * n_ex_nz_mpos / n_ex_nz:.2f}%' if n_ex_nz else 'no excel nz')
print(f'Excel zero without MPOS row: {n_ex_z_nompos:,} / {n_ex_z:,} = {100.0 * n_ex_z_nompos / n_ex_z:.2f}%' if n_ex_z else 'no excel zero')

both = cmp.loc[cmp['bucket'] == 'both_nz'].copy()
if len(both):
    exact_pct = 100.0 * both['exact'].mean()
    print(f'both_nz exact |d|<{EPS}: {int(both["exact"].sum()):,} / {len(both):,} = {exact_pct:.2f}%')
    print('sum excel both_nz=', f"{both['commission_excel'].sum():,.2f}", '| sum mpos=', f"{both['commission_mpos'].sum():,.2f}")

print('\\n=== VERDICT MPOS vs Excel ===')
if n_ex_z and n_ex_z_nompos / n_ex_z >= 0.9 and n_ex_nz and n_ex_nz_mpos / n_ex_nz >= 0.9:
    print('YES: нули отчёта почти не имеют строки MPOS, ненули почти все есть. Гипотеза подтверждается.')
elif n_ex_z and n_ex_z_nompos / n_ex_z >= 0.7:
    print('LIKELY: большая часть Excel=0 без строки MPOS. Добить бакет excel_nz_no_mpos.')
else:
    print('NOT ONLY ZEROS: смотри excel_nz_no_mpos — это реальный недогруз, не нули.')


In [ ]:
print('=== TOP Excel≠0 без MPOS ===')
miss = (
    cmp.loc[cmp['bucket'] == 'excel_nz_no_mpos']
    .sort_values('commission_excel', ascending=False)
    [['inn_key', 'agr_id_key', 'tariff', 'commission_excel']]
    .head(20)
)
display(miss)

print('=== TOP |delta| на пересечении ненулей ===')
top_delta = (
    both.assign(abs_delta=both['delta'].abs())
    .sort_values('abs_delta', ascending=False)
    [['inn_key', 'agr_id_key', 'tariff', 'commission_excel', 'commission_mpos', 'delta']]
    .head(20)
) if len(both) else pd.DataFrame()
display(top_delta)


## 5) ЦФТ DOG_OPER — профиль видов и отбор только месячной

Не берём все `VID_COMISS`. Сначала профиль, затем автоотбор видов, которые бьются с **ненулевой** месячной Excel (не с оборотом).


In [ ]:
sql_cft = f'''
select
  regexp_replace(trim(cast(cl.c_inn as string)), '[^0-9]', '') as inn,
  cast(m.id as string) as agr_id,
  cast(vc.id as string) as vid_id,
  cast(vc.c_name as string) as commis_type,
  cast(o.c_date_create as timestamp) as c_date_create,
  cast(o.c_pay_summ as double) as c_pay_summ,
  cast(o.c_calc_summ as double) as c_calc_summ
from ods.scd1_z_R2_IP_DOG_OPER o
join ods.scd1_z_R2_VID_COMISS vc on vc.id = o.c_vid_comiss
join ods.scd1_z_r2_ip_merchants m on m.id = o.c_parent_id
join ods.scd1_z_client cl on m.c_cl_org = cl.id
where o.c_parent_class = 'R2_IP_MERCHANTS'
  and cl.class_id = 'CL_ORG'
  and cast(o.c_date_create as date) >= cast('{MONTH_START}' as date)
  and cast(o.c_date_create as date) < cast('{MONTH_END_EXCL}' as date)
'''
try:
    cft_raw = fetch_imp(sql_cft, 'CFT DOG_OPER June')
except Exception as exc:
    print('mixed-case fail, try lowercase:', type(exc).__name__, exc)
    sql_cft = sql_cft.replace('scd1_z_R2_IP_DOG_OPER', 'scd1_z_r2_ip_dog_oper').replace('scd1_z_R2_VID_COMISS', 'scd1_z_r2_vid_comiss')
    cft_raw = fetch_imp(sql_cft, 'CFT DOG_OPER June lc')

cft = cft_raw.copy()
cft['inn_key'] = cft['inn'].map(normalize_inn_q1)
cft['agr_id_key'] = cft['agr_id'].map(normalize_agr_q1)
cft['c_calc_summ'] = pd.to_numeric(cft['c_calc_summ'], errors='coerce')
cft['c_pay_summ'] = pd.to_numeric(cft['c_pay_summ'], errors='coerce')
cft = cft.dropna(subset=['agr_id_key'])

excel_nz_agr = set(ex.loc[ex['excel_nz'], 'agr_id_key'])
excel_all_agr = set(ex['agr_id_key'])

type_rows = []
for tname, g in cft.groupby('commis_type', dropna=False):
    agrs = set(g['agr_id_key'])
    inter_nz = agrs & excel_nz_agr
    inter_all = agrs & excel_all_agr
    type_rows.append({
        'commis_type': tname,
        'vid_id': ','.join(sorted({str(x) for x in g['vid_id'].dropna().unique()})[:6]),
        'rows': len(g),
        'agr': len(agrs),
        'sum_calc': g['c_calc_summ'].sum(),
        'sum_pay': g['c_pay_summ'].sum(),
        'recall_excel_nz': round(100.0 * len(inter_nz) / len(excel_nz_agr), 2) if excel_nz_agr else np.nan,
        'precision_vs_excel_nz': round(100.0 * len(inter_nz) / len(agrs), 2) if agrs else np.nan,
        'cover_all_excel': round(100.0 * len(inter_all) / len(excel_all_agr), 2) if excel_all_agr else np.nan,
    })
type_prof = pd.DataFrame(type_rows).sort_values(['recall_excel_nz', 'sum_calc'], ascending=False)
print('=== VID_COMISS vs Excel monthly nonzero ===')
display(type_prof)

name_needles = ('месяч', 'фикс', 'аренд', 'абонент', 'обслуж')
ops_needles = ('процент', 'оборот', 'операц', 'мпс', 'irf', 'интерч')

def looks_monthly(name):
    s = '' if pd.isna(name) else str(name).lower()
    if any(n in s for n in ops_needles):
        return False
    return any(n in s for n in name_needles)

auto = type_prof.loc[
    (type_prof['recall_excel_nz'] >= 40)
    & (type_prof['cover_all_excel'] <= 45)
    & (type_prof['precision_vs_excel_nz'] >= 35)
]
name_hit = type_prof.loc[type_prof['commis_type'].map(looks_monthly)]
picked = pd.concat([auto, name_hit]).drop_duplicates('commis_type')

if CFT_MONTHLY_TYPES_FORCE:
    monthly_types = list(CFT_MONTHLY_TYPES_FORCE)
    print('CFT types FORCED:', monthly_types)
else:
    monthly_types = [t for t in picked['commis_type'].tolist() if pd.notna(t)]
    print('CFT types AUTO monthly:', monthly_types)

if not monthly_types:
    print('WARN: автоотбор пуст. Не беру все виды. Выбери commis_type из таблицы выше и пропиши CFT_MONTHLY_TYPES_FORCE.')
    cft_m = cft.iloc[0:0].copy()
else:
    cft_m = cft.loc[cft['commis_type'].isin(monthly_types)].copy()

cft_map = (
    cft_m.groupby(['inn_key', 'agr_id_key'], as_index=False)
    .agg(
        cft_rows=('c_date_create', 'size'),
        commission_cft_calc=('c_calc_summ', 'sum'),
        commission_cft_pay=('c_pay_summ', 'sum'),
        commis_types=('commis_type', lambda s: ' | '.join(sorted({str(x) for x in s.dropna()}))),
    )
) if len(cft_m) else pd.DataFrame(columns=['inn_key', 'agr_id_key', 'cft_rows', 'commission_cft_calc', 'commission_cft_pay', 'commis_types'])
cft_map['cft_nz'] = is_nz(cft_map['commission_cft_calc']) | is_nz(cft_map['commission_cft_pay'])
print('CFT monthly keys=', f'{len(cft_map):,}', '| rows=', f'{len(cft_m):,}')
print('sum c_calc=', f"{pd.to_numeric(cft_map.get('commission_cft_calc'), errors='coerce').fillna(0).sum():,.2f}")


## 6) Июнь 1:1 — Excel × MPOS × ЦФТ (только месячная)


In [ ]:
tri = ex.merge(
    mpos_map[['inn_key', 'agr_id_key', 'commission_mpos', 'mpos_nz']],
    on=['inn_key', 'agr_id_key'],
    how='left',
).merge(
    cft_map[['inn_key', 'agr_id_key', 'commission_cft_calc', 'commission_cft_pay', 'cft_nz', 'commis_types']],
    on=['inn_key', 'agr_id_key'],
    how='left',
)
tri['mpos_present'] = tri['commission_mpos'].notna()
tri['cft_present'] = tri['cft_nz'].fillna(False) | tri['commission_cft_calc'].notna()
tri['cft_nz'] = tri['cft_nz'].fillna(False)
tri['excel_gross'] = tri['commission_excel'] * VAT
tri['delta_cft_vs_excel_gross'] = pd.to_numeric(tri['commission_cft_calc'], errors='coerce') - tri['excel_gross']

def presence(r):
    if r['mpos_present'] and r['cft_present']:
        return 'both'
    if r['mpos_present']:
        return 'only_mpos'
    if r['cft_present']:
        return 'only_cft'
    return 'neither'

tri['presence'] = tri.apply(presence, axis=1)

print('=== Наличие месячной на периметре Excel June ===')
pres = (
    tri.groupby('presence', as_index=False)
    .agg(keys=('agr_id_key', 'nunique'), inn=('inn_key', 'nunique'), sum_excel=('commission_excel', 'sum'))
)
display(pres)
n = len(tri)
for p in ['both', 'only_mpos', 'only_cft', 'neither']:
    k = int((tri['presence'] == p).sum())
    print(f'  {p}: {k:,} ({100.0 * k / n:.2f}%)')

print('\\n=== Покрытие ненулей Excel ===')
nz = tri.loc[tri['excel_nz']]
print(f'Excel nz keys={len(nz):,}')
print(f'  +MPOS {int(nz["mpos_present"].sum()):,} ({100.0 * nz["mpos_present"].mean():.2f}%)')
print(f'  +CFT monthly {int(nz["cft_present"].sum()):,} ({100.0 * nz["cft_present"].mean():.2f}%)')
print(f'  +оба {int((nz["mpos_present"] & nz["cft_present"]).sum()):,}')

print('\\n=== Покрытие нулей Excel ===')
zz = tri.loc[~tri['excel_nz']]
print(f'Excel zero keys={len(zz):,}')
print(f'  MPOS present {int(zz["mpos_present"].sum()):,} ({100.0 * zz["mpos_present"].mean():.2f}%)')
print(f'  CFT monthly present {int(zz["cft_present"].sum()):,} ({100.0 * zz["cft_present"].mean():.2f}%)')

cft_on_nz = nz.loc[nz['cft_present']].copy()
if len(cft_on_nz):
    cft_exact = np.isclose(
        pd.to_numeric(cft_on_nz['commission_cft_calc'], errors='coerce').fillna(0),
        cft_on_nz['excel_gross'].fillna(0),
        atol=EPS,
        rtol=0,
    )
    print(f'\\nCFT calc vs Excel×{VAT} на Excel-nz: exact {int(cft_exact.sum()):,}/{len(cft_on_nz):,} = {100.0 * cft_exact.mean():.2f}%')


## 7) VERDICT + Excel


In [ ]:
print('=== VERDICT ===')
print(f'Excel June keys={n_ex:,} | nonzero={n_ex_nz:,} ({100.0 * n_ex_nz / n_ex:.2f}%) | zero={n_ex_z:,}')
print(f'MPOS mapped keys={len(mpos_map):,} | raw zero rows in MPOS_RENT: see section 2')
print(f'CFT monthly types={monthly_types}')
print(f'CFT monthly keys={len(cft_map):,}')
print()
print('Гипотеза «нули не пишутся в MPOS»:')
print(f'  Excel zero без строки MPOS = {n_ex_z_nompos:,}/{n_ex_z:,}')
print(f'  Excel nonzero со строкой MPOS = {n_ex_nz_mpos:,}/{n_ex_nz:,}')
print()
print('Если nonzero Excel ≈ 20–21% всех строк отчёта — это и есть «мало клиентов в MPOS».')
print('MPOS тогда 1:1 с ненулевой месячной отчёта, а не со всеми договорами отчёта.')

out = OUT_DIR / f'june_mpos_excel_cft_{MONTH}.xlsx'
with pd.ExcelWriter(out, engine='openpyxl') as w:
    ex.to_excel(w, sheet_name='excel', index=False)
    mpos_raw_stat.to_excel(w, sheet_name='mpos_raw_stat', index=False)
    mpos_map.to_excel(w, sheet_name='mpos_map', index=False)
    buck.to_excel(w, sheet_name='excel_mpos_buckets', index=False)
    type_prof.to_excel(w, sheet_name='cft_vid_score', index=False)
    tri.to_excel(w, sheet_name='excel_mpos_cft', index=False)
    miss.to_excel(w, sheet_name='excel_nz_no_mpos_top', index=False)
print('Saved:', out)
